In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("flax-community/papuGaPT2")
model = AutoModelForCausalLM.from_pretrained("flax-community/papuGaPT2").to(device).eval()

In [ ]:
vocab = [tokenizer.decode([i]) for i in range(len(tokenizer))]

def mask(pred):
    return torch.tensor([pred(t) for t in vocab], device=device)

word_part = mask(lambda t: t.isalpha() and t.islower())
comma = mask(lambda t: t == ",")
end = mask(lambda t: t in [".", "!", "?"])

def word_start(letter):
    return mask(lambda t: t[:1] == " " and t[1:].isalpha() and t[1:].islower() and t[1] == letter)

In [ ]:
def top_k_top_p(logits, k, p):
    values, idx = torch.topk(logits, k)
    probs = torch.softmax(values, -1)
    keep = probs.cumsum(-1) - probs < p
    out = torch.full_like(logits, -float("inf"))
    out[idx[keep]] = values[keep]
    return out

@torch.no_grad()
def generate(prefix, starts, k=50, p=0.9, max_tokens=25):
    ids = tokenizer(tokenizer.eos_token + prefix, return_tensors="pt").input_ids.to(device)
    logprobs = []
    for i in range(max_tokens):
        logits = model(ids).logits[0, -1]
        logp = torch.log_softmax(logits, -1)

        if vocab[ids[0, -1]] == ",":
            allowed = starts
        else:
            allowed = starts | word_part | comma
            if i >= 3:
                allowed = allowed | end
        logits = logits.masked_fill(~allowed, -float("inf"))
        logits[ids[0]] -= 2  # kara za powtórzenia

        next_id = torch.multinomial(torch.softmax(top_k_top_p(logits, k, p), -1), 1)
        logprobs.append(logp[next_id].item())
        ids = torch.cat([ids, next_id[None]], 1)
        if vocab[next_id] in [".", "!", "?"]:
            return tokenizer.decode(ids[0, 1:]), logprobs
    return None  # nie skończyło się kropką

def best_sentence(prefix, n=10):
    starts = word_start(prefix[0].lower())
    results = [r for r in (generate(prefix, starts) for _ in range(n)) if r]
    return max(results, key=lambda r: sum(r[1]) / len(r[1]) + 0.5 * min(r[1]))[0]

In [ ]:
PREFIXES = [
    "Prawdziwy piekarz przyprawia pieczywo pieprzem",
    "Kasia kupiła kolorowe",
    "Stary sąsiad sprzedał",
    "Wesoły wędkarz wyciągnął",
    "Mój mąż miał",
    "Cichy chłopiec chował",
    "Nasz nowy nauczyciel",
    "Dzielny dziadek dał",
]

for prefix in PREFIXES:
    print(best_sentence(prefix))